In [2]:
import os, json
tmp_dir = '/data/home/scv7387/run/tv_series_plus/3D-Speaker/docs/可视化/temp'
vad_json_path = os.path.join(tmp_dir, 'cluster_results_vision_vad_processed_for_HMM_nested_X_uniq.json')
audio_obs_json_path = os.path.join(tmp_dir, 'cluster_results_audio_processed_for_HMM_nested_X.json')
audio_dec_json_path = os.path.join(tmp_dir, 'cluster_results_audio_vision_vad_nested_hmm_full_adur_grps_has_neg1(unreliable_pp=100).json')
audio_smoothed_json_path = os.path.join(tmp_dir, 'cluster_results_audio_vision_vad_nested_hmm_full_adur_grps_has_neg1(unreliable_pp=5).json')

# check accuracy of part of vad samples
## the part have the same label in both vad and audio clustering results
with open(vad_json_path, 'r', encoding='utf-8') as f:
    vad_cluster_results = json.load(f)
with open(audio_obs_json_path, 'r', encoding='utf-8') as f:
    audio_obs_results = json.load(f)
with open(audio_dec_json_path, 'r', encoding='utf-8') as f:
    audio_decode_results = json.load(f)
with open(audio_smoothed_json_path, 'r', encoding='utf-8') as f:
    audio_smoothed_results = json.load(f)
print(f"Total number of samples in VAD clustering results: {len(vad_cluster_results)}")

keys_selected_obs = []
for key in vad_cluster_results:
    if vad_cluster_results[key] == audio_obs_results[key]:
        keys_selected_obs.append(key)
print(f"Number of selected samples for audio obs: {len(keys_selected_obs)}")
with open(os.path.join(tmp_dir, 'selected_keys_for_audio_obs.json'), 'w', encoding='utf-8') as f:
    json.dump({k: vad_cluster_results[k] for k in keys_selected_obs}, f, indent=2)

keys_selected_decode = []
for key in vad_cluster_results:
    if vad_cluster_results[key] == audio_decode_results[key]:
        keys_selected_decode.append(key)
print(f"Number of selected samples for audio decode: {len(keys_selected_decode)}")  # 数量更多，因为更充分利用了vad信息
with open(os.path.join(tmp_dir, 'selected_keys_for_audio_decode(unreliable_pp=100).json'), 'w', encoding='utf-8') as f:
    json.dump({k: vad_cluster_results[k] for k in keys_selected_decode}, f, indent=2)

keys_selected_smoothed = []
for key in vad_cluster_results:
    if vad_cluster_results[key] == audio_smoothed_results[key]:
        keys_selected_smoothed.append(key)
print(f"Number of selected samples for audio smoothed: {len(keys_selected_smoothed)}")
with open(os.path.join(tmp_dir, 'selected_keys_for_audio_smoothed(unreliable_pp=5).json'), 'w', encoding='utf-8') as f:
    json.dump({k: vad_cluster_results[k] for k in keys_selected_smoothed}, f, indent=2)

Total number of samples in VAD clustering results: 1261
Number of selected samples for audio obs: 1195
Number of selected samples for audio decode: 1218
Number of selected samples for audio smoothed: 1198


In [10]:
(0.9335*1261-0.9728*1198)/(1261-1198)  # 计算未采纳vad信息的样本的准确率
# (0.9335*1261-0.9626*1218)/(1261-1218)  # 计算未采纳vad信息的样本的准确率


0.18617619047619074

结果显示，从能根据vad确定标签的样本集合中，筛选标签与initialization中使用的hmm observation一致的样本子集，其准确率可达0.9728。可以把这部分样本视作pseudo validation set，用于后续模型选择。

In [32]:
import numpy as np
def calculate_acc_fromdic(pred_dic, ref_dic):
    correct_num = sum(list(map(lambda k: 1 if pred_dic[k] == ref_dic[k] else 0, ref_dic.keys())))
    ref_total = len(ref_dic)
    return correct_num / ref_total if ref_total > 0 else 0

def calculate_acc_fromjson(pred_json_path, ref_json_path=os.path.join(tmp_dir, 'selected_keys_for_audio_obs.json')):
    with open(pred_json_path, 'r', encoding='utf-8') as f:
        pred_dic = json.load(f)
    with open(ref_json_path, 'r', encoding='utf-8') as f:
        ref_dic = json.load(f)
    return calculate_acc_fromdic(pred_dic, ref_dic)

def get_json_paths_ft(round_dir):
    ft_dirs = [d for d in os.listdir(round_dir) if os.path.isdir(os.path.join(round_dir, d)) and d.startswith("ft_epoch")]
    json_paths_ft = [os.path.join(round_dir, d, 'pseudo_labels_audio_pred.json') for d in ft_dirs]
    return json_paths_ft

def get_json_paths_round(exp_dir):
    round_dirs = [d for d in os.listdir(exp_dir) if os.path.isdir(os.path.join(exp_dir, d))]
    json_paths_round = [os.path.join(exp_dir, d, 'pseudo_label', 'pseudo_labels_audio_nested_hmm_full(unreliable_pp=5.0).json') for d in round_dirs]
    return json_paths_round

exp_dir = '/data/home/scv7387/run/tv_series_plus/3D-Speaker/egs/3dspeaker/speaker-diarization/runs/the big bang theory/exp_video/result/self_supervised/2. 仅微调dense layer，从hid_feat构建数据集/exp1'
# 检查ref_json_path是否能用于确定 best epoch
round_dirs = [d for d in os.listdir(exp_dir) if os.path.isdir(os.path.join(exp_dir, d)) and d.startswith("round")]
round_dirs.sort()
for round_dir in round_dirs:
    json_paths = get_json_paths_ft(os.path.join(exp_dir, round_dir))
    json_paths.sort()
    epochs_names = [os.path.basename(os.path.dirname(p)) for p in json_paths]
    accs = np.array([calculate_acc_fromjson(p) for p in json_paths])
    print(f"All epochs in round {round_dir}: {epochs_names}")
    print(f"Round {round_dir} accs: {accs}")
    best_epoch = epochs_names[np.argmax(accs)]
    print(f"Round {round_dir}, Best epoch: {best_epoch}, best acc: {max(accs)}")

print("=========================================")
# 检查ref_json_path是否能用于确定 best round
json_paths = get_json_paths_round(exp_dir)
json_paths.sort()
rounds_names = [os.path.basename(os.path.dirname(os.path.dirname(p))) for p in json_paths]
accs_rounds = np.array([calculate_acc_fromjson(p) for p in json_paths])
print(f"All rounds: {rounds_names}")
print(f"Rounds accs: {accs_rounds}")
best_json_path_round = json_paths[np.argmax(accs_rounds)]
best_round = os.path.basename(os.path.dirname(os.path.dirname(best_json_path_round)))
print(f"Best round: {best_round}, best acc: {max(accs_rounds)}")

All epochs in round round0: ['ft_epoch_0', 'ft_epoch_1', 'ft_epoch_2', 'ft_epoch_3', 'ft_epoch_4', 'ft_epoch_5', 'ft_epoch_6', 'ft_epoch_7', 'ft_epoch_8', 'ft_epoch_9']
Round round0 accs: [0.88033473 0.97405858 0.97573222 0.98661088 0.98912134 0.9790795
 0.95062762 0.97405858 0.97238494 0.86359833]
Round round0, Best epoch: ft_epoch_4, best acc: 0.9891213389121339
All epochs in round round1: ['ft_epoch_0', 'ft_epoch_1', 'ft_epoch_2', 'ft_epoch_3', 'ft_epoch_4', 'ft_epoch_5', 'ft_epoch_6', 'ft_epoch_7']
Round round1 accs: [0.98158996 0.95564854 0.97740586 0.93807531 0.9916318  0.92468619
 0.9832636  0.94560669]
Round round1, Best epoch: ft_epoch_4, best acc: 0.9916317991631799
All epochs in round round2: ['ft_epoch_0', 'ft_epoch_1', 'ft_epoch_2', 'ft_epoch_3', 'ft_epoch_4', 'ft_epoch_5', 'ft_epoch_6', 'ft_epoch_7']
Round round2 accs: [0.97824268 0.98158996 0.97824268 0.97405858 0.97322176 0.98661088
 0.95983264 0.98158996]
Round round2, Best epoch: ft_epoch_5, best acc: 0.98661087866108

该方法选择的epoch与原有通过validation set 选择epoch的结果较为接近，可以尝试作为替换方案（也可以固定下来，用 5 先跑一版）。

round选择方面，考虑确定为5-6。

unreliable_pp选择方面，也考虑确定下来。动态调整效果并不好。

下面尝试对于model给出的prediction，也仅选择性采纳：

In [20]:
import os, json
import numpy as np
unreliable_pp = 5.0
pseudo_label_dir = '/data/home/scv7387/run/tv_series_plus/3D-Speaker/egs/3dspeaker/speaker-diarization/runs/the big bang theory/exp_video/result/self_supervised/2. 仅微调dense layer，从hid_feat构建数据集/exp6/initial/pseudo_label'
# Load unreliable segment IDs and initial cluster results
audio_seg_ids = np.load(os.path.join(pseudo_label_dir, 'audio_seg_ids.npy'), allow_pickle=True)
audio_cluster_unreliable_metrics = np.load(os.path.join(pseudo_label_dir, 'alabels_unreliable_metrics.npy'), allow_pickle=True)
idxs_unreliable = np.argsort(audio_cluster_unreliable_metrics)[:int(unreliable_pp / 100 * len(audio_seg_ids))]
audio_seg_ids_unreliable = audio_seg_ids[idxs_unreliable]

audio_cluster_result_files = [f for f in os.listdir(pseudo_label_dir) if f.endswith('.json') and 'cluster_results_audio_processed' in f]
assert len(audio_cluster_result_files) == 1, f"No or multiple cluster_result_processed file found in {pseudo_label_dir}: {audio_cluster_result_files}"
cluster_result_file = os.path.join(pseudo_label_dir, audio_cluster_result_files[0])
with open(cluster_result_file, 'r') as f:
    initial_cluster_results = json.load(f)

print(type(audio_seg_ids_unreliable))
print('E02-199' in audio_seg_ids_unreliable)
print('E02-199' in audio_seg_ids_unreliable.tolist())

<class 'numpy.ndarray'>
True
True


In [24]:
with open("/data/home/scv7387/run/tv_series_plus/3D-Speaker/egs/3dspeaker/speaker-diarization/runs/the big bang theory/exp_video/json/subseg.json", 'r') as f:
    subseg_info = json.load(f)
sample_ids = list(subseg_info.keys())
for sid in sample_ids:
      if sid in audio_seg_ids_unreliable:
            print(sid)

import pickle
with open("/data/home/scv7387/run/tv_series_plus/3D-Speaker/egs/3dspeaker/speaker-diarization/runs/the big bang theory/exp_video/result/self_supervised/2. 仅微调dense layer，从hid_feat构建数据集/exp6/round0/pseudo_label/alabels_unreliable_dic.pkl", 'rb') as f:
            alabels_unreliable_dic = pickle.load(f)
for key in alabels_unreliable_dic.keys():
    if alabels_unreliable_dic[key] == 2:
        print(key)

E01-48
E01-101
E01-117
E01-204
E01-215
E01-266
E01-292
E01-298
E01-389
E01-390
E01-416
E02-15
E02-45
E02-102
E02-115
E02-118
E02-125
E02-135
E02-173
E02-182
E02-191
E02-192
E02-194
E02-198
E02-199
E02-262
E02-265
E02-287
E03-7
E03-20
E03-21
E03-27
E03-37
E03-40
E03-41
E03-42
E03-66
E03-72
E03-98
E03-182
E03-184
E03-193
E03-218
E03-226
E03-259
E03-301
E03-334
E03-349
E03-350
E04-17
E04-41
E04-47
E04-48
E04-50
E04-52
E04-59
E04-76
E04-189
E04-196
E04-339
E04-348
E04-356
E04-365
E05-6
E05-13
E05-17
E05-37
E05-50
E05-68
E05-111
E05-118
E05-125
E05-143
E05-216
E05-219
E05-225
E05-240
E05-241
E05-255
E05-264
E05-267
E05-272
E05-338
E05-364
E06-52
E06-60
E06-66
E06-153
E06-163
E06-176
E06-198
E06-199
E06-203
E06-223
E06-236
E06-251
E06-257
E06-258
E06-262
E06-264
E06-277
E06-289
E06-291
E06-294
E06-297
E06-301
E06-346
E06-348
E06-353
E06-361
E06-376
E07-23
E07-83
E07-92
E07-146
E07-165
E07-170
E07-203
E07-206
E07-234
E07-242
E07-249
E07-281
E07-359
E07-366
E07-368
E07-369
E07-370
E07-372
E07-